In [11]:
import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio

def arc_segment(p0, p1, n=200):
    t = np.linspace(0, 1, n)
    z = (1 - t) * p0 + t * p1
    return z.real, z.imag

def arc_semicircle(center, radius, a0, a1, n=300):
    t = np.linspace(a0, a1, n)
    z = center + radius * np.exp(1j * t)
    return z.real, z.imag

def concat_paths(*paths):
    xs, ys = [], []
    for k, (x, y) in enumerate(paths):
        if k == 0:
            xs.append(x); ys.append(y)
        else:
            xs.append(x[1:]); ys.append(y[1:])
    return np.concatenate(xs), np.concatenate(ys)

# -----------------------------
# Curva compuesta Γ = Γ1 ∪ Γ2 ∪ Γ3
# -----------------------------
x1, y1 = arc_segment(0 + 0j, 1 + 0j, n=220)
x2, y2 = arc_semicircle(center=0 + 0j, radius=1.0, a0=0.0, a1=np.pi, n=420)  # de (1,0) a (-1,0)
x3, y3 = arc_segment(-1 + 0j, -1 - 0.7j, n=180)

X, Y = concat_paths((x1, y1), (x2, y2), (x3, y3))

# -----------------------------
# Puntos especiales
# -----------------------------
# Endpoints de cada arco
E1 = np.array([0+0j, 1+0j])          # Γ1
E2 = np.array([1+0j, -1+0j])         # Γ2
E3 = np.array([-1+0j, -1-0.7j])      # Γ3

# Todos los endpoints de arcos (incluye knots)
all_endpoints = np.unique(np.concatenate([E1, E2, E3]))

# Clasificación por "grado" (cuántos arcos inciden)
# aquí, por construcción: 0 y -1-0.7i son endpoints globales; ±1 son knots
global_endpoints = np.array([0+0j, -1-0.7j])
knots = np.array([1+0j, -1+0j])

gx, gy = global_endpoints.real, global_endpoints.imag
kx, ky = knots.real, knots.imag

# -----------------------------
# GIF config (cíclico: vuelve al inicio)
# -----------------------------
gif_path = "composed_curve.gif"
fps = 30
n_frames = 180

idx = np.linspace(0, len(X) - 1, n_frames, endpoint=False).astype(int)
idx = np.r_[idx, idx[0]]  # último frame = primero (salto suave)

pad = 0.2
xmin, xmax = X.min() - pad, X.max() + pad
ymin, ymax = Y.min() - pad, Y.max() + pad

# Colores
global_endpoint_color = "dodgerblue"
knot_color = "dodgerblue"
moving_color = "black"

frames = []
fig, ax = plt.subplots(figsize=(6, 6), dpi=120)

for i in idx:
    ax.clear()

    # curva
    ax.plot(x1, y1, linewidth=2, label=r"$\Gamma_1$")
    ax.plot(x2, y2, linewidth=2, label=r"$\Gamma_2$")
    ax.plot(x3, y3, linewidth=2, label=r"$\Gamma_3$")

    # puntos: endpoints globales y knots (1,0), (-1,0)
    ax.scatter(gx, gy, s=95, color=global_endpoint_color)#, zorder=6, label="Puntos Extremos")
    ax.scatter(kx, ky, s=95, color=knot_color, zorder=6, label="Puntos Extremos")

    # punto móvil
    ax.scatter([X[i]], [Y[i]], s=95, color=moving_color, zorder=7)

    ax.axhline(0, linewidth=0.8)
    ax.axvline(0, linewidth=0.8)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel("Re")
    ax.set_ylabel("Im")
    ax.set_title("Curva compuesta")
    ax.legend(loc="upper right", fontsize=9)

    # Render robusto
    fig.canvas.draw()
    rgba = np.asarray(fig.canvas.buffer_rgba())
    frames.append(rgba[:, :, :3].copy())

plt.close(fig)

imageio.mimsave(gif_path, frames, fps=fps, loop=0)
print(f"GIF guardado en: {gif_path}")

GIF guardado en: composed_curve.gif
